#Import libraries

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import time
from skimage.feature import local_binary_pattern
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

#Camera

In [ ]:

# =========================================================
# 1. SETTINGS
# =========================================================

CAMERA_URL = "http://xxx.xxx.x.xxx:xxxx/video"

MODEL_PATH = "oral_cancer_voting_model.pkl"
LANDMARKER_PATH = "face_landmarker.task"

IMAGE_SIZE = (128, 128)

LBP_POINTS = 8
LBP_RADIUS = 1

PROBABILITY_THRESHOLD = 0.40

CLASS_NAMES = {
    0: "Normal",
    1: "Oral Cancer"
}


# =========================================================
# 2. LOAD MODEL
# =========================================================

print("Loading VotingClassifier...")

model = joblib.load(MODEL_PATH)

print("Voting model loaded.")
print("Model classes:", model.classes_)


# =========================================================
# 3. HOG DESCRIPTOR
# =========================================================

hog = cv2.HOGDescriptor(
    (128, 128),
    (16, 16),
    (8, 8),
    (8, 8),
    9
)

print("HOG descriptor size:", hog.getDescriptorSize())


# =========================================================
# 4. FEATURE EXTRACTION: HOG + LBP
# =========================================================

def extract_features(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    gray = cv2.resize(
        gray,
        IMAGE_SIZE
    )

    # HOG features
    hog_features = hog.compute(
        gray,
        winStride=(16, 16),
        padding=(0, 0)
    ).flatten()

    # LBP features
    lbp = local_binary_pattern(
        gray,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        range=(0, 10)
    )

    lbp_hist = lbp_hist.astype(np.float32)

    if lbp_hist.sum() > 0:
        lbp_hist /= lbp_hist.sum()

    # Combine features
    features = np.concatenate(
        [hog_features, lbp_hist]
    ).astype(np.float32)

    return features.reshape(1, -1)


# =========================================================
# 5. CHECK FEATURE DIMENSION
# =========================================================

EXPECTED_FEATURES = 8110

if hog.getDescriptorSize() + 10 != EXPECTED_FEATURES:
    raise ValueError(
        "HOG feature count mismatch. "
        "Check the HOG parameters."
    )

if (
    hasattr(model, "n_features_in_")
    and model.n_features_in_ != EXPECTED_FEATURES
):
    raise ValueError(
        f"Model expects {model.n_features_in_} features, "
        f"but this program creates {EXPECTED_FEATURES}. "
        "Use the same feature extraction settings as training."
    )


# =========================================================
# 6. PREDICTION FUNCTION
# =========================================================

def predict_oral_cancer(roi):

    if roi is None or roi.size == 0:
        return None

    features = extract_features(roi)

    if not hasattr(model, "predict_proba"):
        raise RuntimeError(
            "The loaded model does not support predict_proba()."
        )

    probabilities = model.predict_proba(features)[0]
    classes = list(model.classes_)

    # Find the probability belonging to each class.
    if 0 not in classes or 1 not in classes:
        raise ValueError(
            f"Expected classes 0 and 1, but found {classes}"
        )

    normal_probability = float(
        probabilities[classes.index(0)]
    )

    cancer_probability = float(
        probabilities[classes.index(1)]
    )

    # Convert probabilities to percentages.
    normal_percent = normal_probability * 100.0
    cancer_percent = cancer_probability * 100.0

    # Apply the selected experimental threshold.
    if cancer_probability >= PROBABILITY_THRESHOLD:
        label = "Oral Cancer"
    else:
        label = "Normal"

    return {
        "label": label,
        "normal_percent": normal_percent,
        "cancer_percent": cancer_percent
    }


# =========================================================
# 7. MEDIAPIPE FACE LANDMARKER
# =========================================================

print("Loading MediaPipe FaceLandmarker...")

base_options = python.BaseOptions(
    model_asset_path=LANDMARKER_PATH
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1
)

landmarker = vision.FaceLandmarker.create_from_options(
    options
)

print("FaceLandmarker loaded.")


# Mouth landmarks
MOUTH_IDS = [
    61, 185, 40, 39, 37, 0,
    267, 269, 270, 409, 291,
    375, 321, 405, 314, 17,
    84, 181, 91, 146, 13, 14
]


# =========================================================
# 8. HELPER FUNCTIONS
# =========================================================

def get_point(landmarks, index, width, height):

    x = int(landmarks[index].x * width)
    y = int(landmarks[index].y * height)

    return x, y


def clamp(value, low, high):

    return max(low, min(value, high))


def get_mouth_roi(frame, landmarks):

    height, width = frame.shape[:2]

    points = np.array([
        get_point(landmarks, i, width, height)
        for i in MOUTH_IDS
    ])

    x_min = int(points[:, 0].min())
    x_max = int(points[:, 0].max())

    y_min = int(points[:, 1].min())
    y_max = int(points[:, 1].max())

    mouth_width = max(1, x_max - x_min)

    # Expand ROI around the lips.
    x_min = int(x_min - 0.10 * mouth_width)
    x_max = int(x_max + 0.10 * mouth_width)

    y_min = int(y_min - 0.15 * mouth_width)
    y_max = int(y_max + 0.20 * mouth_width)

    x_min = clamp(x_min, 0, width - 1)
    x_max = clamp(x_max, 1, width)

    y_min = clamp(y_min, 0, height - 1)
    y_max = clamp(y_max, 1, height)

    if x_max <= x_min or y_max <= y_min:
        return None, None

    roi = frame[y_min:y_max, x_min:x_max].copy()

    box = (x_min, y_min, x_max, y_max)

    return roi, box


# =========================================================
# 9. POSSIBLE TONGUE COLOR MASK
# =========================================================

def make_tongue_mask(roi):

    if roi is None or roi.size == 0:
        return None

    hsv = cv2.cvtColor(
        roi,
        cv2.COLOR_BGR2HSV
    )

    lower_red1 = np.array([0, 40, 40])
    upper_red1 = np.array([15, 255, 255])

    lower_red2 = np.array([160, 40, 40])
    upper_red2 = np.array([179, 255, 255])

    mask1 = cv2.inRange(
        hsv,
        lower_red1,
        upper_red1
    )

    mask2 = cv2.inRange(
        hsv,
        lower_red2,
        upper_red2
    )

    mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    return mask


# =========================================================
# 10. OPEN DROIDCAM
# =========================================================

print("Connecting to DroidCam...")

cap = cv2.VideoCapture(
    CAMERA_URL,
    cv2.CAP_FFMPEG
)

if not cap.isOpened():

    cap.release()
    landmarker.close()

    raise RuntimeError(
        "DroidCam could not be opened. "
        "Check the IP address, port, and phone connection."
    )

# Wait for the first usable frame.
first_frame_received = False

for _ in range(20):

    ret, test_frame = cap.read()

    if ret and test_frame is not None:
        first_frame_received = True
        break

    time.sleep(0.2)

if not first_frame_received:

    cap.release()
    landmarker.close()

    raise RuntimeError(
        "DroidCam opened, but no video frame was received."
    )

print("DroidCam connected.")


# =========================================================
# 11. MAIN LOOP
# =========================================================

timestamp_ms = 0

try:

    while True:

        ret, frame = cap.read()

        if not ret or frame is None:
            print("Failed to receive frame.")
            break

        display_frame = frame.copy()

        # Convert image to RGB for MediaPipe.
        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = max(
            timestamp_ms + 1,
            int(time.monotonic() * 1000)
        )

        result = landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        roi = None
        box = None
        mask = None
        prediction = None

        # -------------------------------------------------
        # DETECT FACE AND MOUTH
        # -------------------------------------------------

        if result.face_landmarks:

            landmarks = result.face_landmarks[0]

            roi, box = get_mouth_roi(
                frame,
                landmarks
            )

            if roi is not None and roi.size > 0:

                x1, y1, x2, y2 = box

                cv2.rectangle(
                    display_frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

                # -------------------------------------------------
                # PREDICT CLASS PROBABILITIES
                # -------------------------------------------------

                prediction = predict_oral_cancer(roi)

                if prediction is not None:

                    normal_percent = prediction[
                        "normal_percent"
                    ]

                    cancer_percent = prediction[
                        "cancer_percent"
                    ]

                    label = prediction["label"]

                    # Show Normal probability separately.
                    cv2.putText(
                        display_frame,
                        f"Normal: {normal_percent:.2f}%",
                        (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        (0, 255, 0),
                        2,
                        cv2.LINE_AA
                    )

                    # Show Oral Cancer probability separately.
                    cv2.putText(
                        display_frame,
                        f"Oral Cancer: {cancer_percent:.2f}%",
                        (10, 70),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        (0, 0, 255),
                        2,
                        cv2.LINE_AA
                    )

                    # Show selected class.
                    cv2.putText(
                        display_frame,
                        f"Model result: {label}",
                        (10, 105),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        (0, 255, 255),
                        2,
                        cv2.LINE_AA
                    )

                mask = make_tongue_mask(roi)

        else:

            cv2.putText(
                display_frame,
                "No face detected - face the camera",
                (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 0, 255),
                2,
                cv2.LINE_AA
            )

        # -------------------------------------------------
        # SHOW WINDOWS
        # -------------------------------------------------

        cv2.imshow(
            "DroidCam - Oral Cancer ML",
            display_frame
        )

        if roi is not None and roi.size > 0:

            cv2.imshow(
                "Mouth ROI",
                cv2.resize(roi, (320, 240))
            )

        if mask is not None:

            cv2.imshow(
                "Possible Tongue Color Mask",
                mask
            )

        # Press Q to exit.
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break


finally:

    cap.release()
    landmarker.close()
    cv2.destroyAllWindows()

    print("Camera stopped.")


Loading VotingClassifier...
Voting model loaded.
Model classes: [0 1]
HOG descriptor size: 8100
Loading MediaPipe FaceLandmarker...
FaceLandmarker loaded.
Connecting to DroidCam...
DroidCam connected.
Camera stopped.
